<div dir="rtl" align="right">

# ANFIS

درسی از [مدرسه پایتون و هوش مصنوعی](https://www.linkedin.com/in/behrooz-taheri-phd-smieee-b41b26113/)، دکتر بهروز طاهری

**ANFIS** (جانگ، ۱۹۹۳) سیستم سوگنو را به شکل شبکه‌ای پنج‌لایه درمی‌آورد تا از داده یاد بگیرد.

- لایه ۱: درجه عضویت (پارامترهای مقدم)
- لایه ۲: قدرت قاعده با ضرب
- لایه ۳: نرمال‌سازی
- لایه ۴: خروجی قاعده (پارامترهای تالی)
- لایه ۵: جمع نهایی

## یادگیری ترکیبی
در گذر رو به جلو تالی‌ها با کمترین مربعات و در گذر رو به عقب مقدم‌ها با گرادیان به‌روز می‌شوند.

> تعداد قواعد $M^n$ است؛ با ورودی زیاد نفرین ابعاد ظاهر می‌شود. انتخاب ورودی یا خوشه‌بندی کمک می‌کند.

</div>

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
X = rng.uniform(-3, 3, (400, 2))
y = np.sin(X[:, 0]) * np.cos(X[:, 1])

centers = np.tile(np.array([-3.0, 0.0, 3.0]), (2, 1))
sigmas = np.full((2, 3), 1.5)

def design(X, c, s):
    m = np.exp(-(X[:, :, None] - c) ** 2 / (2 * s ** 2))
    w = (m[:, 0, :, None] * m[:, 1, None, :]).reshape(len(X), -1)
    nw = w / w.sum(1, keepdims=True)
    A = np.c_[X, np.ones(len(X))]
    return (nw[:, :, None] * A[:, None, :]).reshape(len(X), -1)

def solve(c, s):
    A = design(X, c, s)
    q = np.linalg.lstsq(A, y, rcond=None)[0]
    return q, float(np.sqrt(np.mean((A @ q - y) ** 2)))

q, e = solve(centers, sigmas)
print("start RMSE", round(e, 4))
lr = 0.1
for epoch in range(40):
    q, e = solve(centers, sigmas)
    g = np.zeros_like(centers)
    for i in np.ndindex(g.shape):
        c2 = centers.copy(); c2[i] += 1e-4
        g[i] = (solve(c2, sigmas)[1] - e) / 1e-4
    trial = centers - lr * g
    if solve(trial, sigmas)[1] < e: centers, lr = trial, lr * 1.2
    else: lr *= 0.5
print("final RMSE", round(solve(centers, sigmas)[1], 4))
print(centers.round(2))